# Bare-bones Transformer — Model Card

Decoder-only Transformer implemented as a plain Python class with raw `torch.Tensor` parameters. No `nn.Module`. No `nn.MultiheadAttention`. Every operation written explicitly.

**Dataset:** TinyShakespeare (~1.1M characters). Split: 90/5/5 train/val/test.

## Hyperparameters

| Parameter | Value |
|---|---|
| d_model | 384 |
| d_ff | 1536 (4× d_model) |
| H (heads) | 8 |
| d_k = d_v | 48 (d_model / H) |
| layers | 6 |
| n (sequence length) | 256 |
| B (batch size) | 32 |
| lr | 3e-4 |
| dropout | 0.1 |
| bpe_merges | 500 (0 = char-level) |

## Optimizer & Training

- **Optimizer:** AdamW (`Beight_decay=0.01`)
- **Scheduler:** ReduceLROnPlateau (`factor=0.5`, `patience=2`)
- **Gradient clipping:** `max_norm=1.0`
- **Early stopping:** patience=5 epochs on val loss; restores best-val checkpoint

## Key Architecture Choices

| Component | Choice |
|---|---|
| Positional encoding | RoPE (applied to Q and K only) |
| Activation | GELU |
| Normalization | Pre-norm LayerNorm (before each sub-layer) |
| Output projection | Weight tying (reuses W_emb^T) |
| Final layer | LayerNorm before output projection |
| Tokenization | BPE or char-level (controlled by `cfg.bpe_merges`) |

## Step 1: Data Loading & Tokenization

1. Load TinyShakespeare dataset from file.
2. **Tokenization** (controlled by `cfg.bpe_merges`):
   - `bpe_merges=0`: character-level. Vocab = sorted unique chars (66 tokens for TinyShakespeare).
   - `bpe_merges=N`: train BPE with N merge operations on the full text, growing vocab from 66 to 66+N. With N=500, average ~2.3 chars/token.
   - Encode: `encode(text, merges, vocab)` → list of integer token IDs
   - Decode: `decode(ids, vocab)` → string
3. Encode full text → 1D tensor of token IDs.
4. Split into non-overlapping chunks of length `n` → shape `(n_chunks, n)`. Final partial chunk discarded.
5. Split chunks into train / val / test (90 / 5 / 5).
6. Returns `avg_chars_per_token` for BPC metric calculation.

## Step 2: Model & Training Loop

### Parameters

| Tensor | Shape | Notes |
|---|---|---|
| W_emb | (vocab_size, d_model) | Token embeddings; also reused as output projection (weight tying) |
| Wq, Wk, Wv, Wo | (d_model, d_model) × layers | Attention projections per layer |
| Wff1 | (d_model, d_ff) × layers | FFN input projection |
| Wff2 | (d_ff, d_model) × layers | FFN output projection |
| gamma_attn, beta_attn | (d_model,) × layers | Pre-attention LayerNorm scale/shift |
| gamma_ffn, beta_ffn | (d_model,) × layers | Pre-FFN LayerNorm scale/shift |
| gamma_out, beta_out | (d_model,) | Final LayerNorm scale/shift |

All weight matrices initialized with `randn * 0.02`. LayerNorm: gamma=1, beta=0. No bias terms.

---

### Forward Pass

**Token lookup:** (B, n) → index into W_emb → **(B, n, d_model)**

---

### Transformer Block (repeated × layers)

#### Attention Sub-layer

1. **Pre-norm LayerNorm** on X
2. **QKV projections:** Q, K, V = X @ Wq/Wk/Wv → each (B, n, d_model)
3. **Split heads:** reshape + transpose → **(B, H, n, d_k)**
4. **RoPE** applied to Q and K (not V): rotates adjacent dimension pairs by position-dependent angles
5. **Scaled dot-product scores:** S = Q @ K^T / sqrt(d_k) → (B, H, n, n)
6. **Causal mask:** add -inf to upper triangle (prevents attending to future tokens)
7. **Softmax** → attention weights A (B, H, n, n)
8. **Dropout** on A
9. **Value aggregation:** O = A @ V → (B, H, n, d_v)
10. **Recombine heads:** transpose + reshape → (B, n, d_model)
11. **Output projection:** X_attn = O @ Wo
12. **Dropout** on X_attn → **residual:** X = X + X_attn

#### Feed-Forward Sub-layer

1. **Pre-norm LayerNorm** on X
2. **Z = X @ Wff1** → (B, n, d_ff)
3. **GELU activation** (tanh approximation)
4. **X_ff = Z_gelu @ Wff2** → (B, n, d_model)
5. **Dropout** on X_ff → **residual:** X = X + X_ff

---

### Output Layer

1. **Final LayerNorm** on X
2. **Weight-tied projection:** logits = X @ W_emb^T → (B, n, vocab_size)
3. **Targets:** input tokens shifted left by 1 (token t predicts token t+1)
4. **Loss:** CrossEntropyLoss — implemented manually with numerically stable log-sum-exp

---

### Optimizer Step (per batch)

```
loss.backward()
clip_grad_norm_(model.params(), max_norm=1.0)
optimizer.step()    # AdamW
optimizer.zero_grad()
```

---

### Checkpointing & Inference

- Training keeps an in-memory snapshot of the best-val-loss parameters; restored at end of run
- `save_model()` writes: `run_TIMESTAMP.pt` (weights) + `_cfg.json` + `_vocab.json` + `_merges.json`
- `load_for_inference(weights=...)` reconstructs model + tokenizer from a checkpoint with no prior setup

### Generation

```python
generate(prompt="...", max_new_tokens=500, temperature=0.8, top_k=50)
```

Left-pads short prompts with token 0 to fill the full context window of length n.
Samples autoregressively: temperature scales logits, top-k masks low-probability tokens before softmax.

In [8]:
import torch
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class Config:
    """Hyperparameters and derived values for the Transformer. d_k and d_v are
    computed automatically from d_model and H. Device is auto-detected."""
    vocab_size: int = 0
    B: int = 32         # batch size
    n: int = 256        # sequence length (context window)
    d_model: int = 384  # embedding / hidden dimension
    d_ff: int = 1536    # feed-forward inner dimension (typically 4 × d_model)
    H: int = 8          # number of attention heads
    epochs: int = 100
    layers: int = 6
    lora_alpha: int = 4
    lora_rank: int = 4
    lr: float = 3e-4
    drop_prob: float = 0.1
    bpe_merges: int = 0  # 0 = char-level; >0 = BPE with this many merges
    device: Optional[torch.device] = None

    def __post_init__(self):
        # d_k = d_model / H; each head attends over a d_k-dimensional subspace
        self.d_k = self.d_model // self.H
        self.d_v = self.d_k
        if self.device is None:
            if torch.backends.mps.is_available():
                self.device = torch.device('mps')
            elif torch.cuda.is_available():
                self.device = torch.device('cuda')
            else:
                self.device = torch.device('cpu')
        print(f"Using device: {self.device}")

In [9]:
import torch
from collections import Counter

ASCII_VOCAB = [chr(i) for i in range(128)]  # stable base vocab for all corpora

def train_bpe(text, num_merges):
    """Learn BPE merge rules from text.

    Starts from the full ASCII vocabulary and greedily merges the most frequent
    adjacent pair at each step. Returns the full vocab (ASCII + merged tokens)
    and the ordered list of merge rules needed to reproduce the encoding.
    """
    vocab = ASCII_VOCAB[:]
    tokens = list(text)
    merges = []

    for _ in range(num_merges):
        pairs = Counter(zip(tokens, tokens[1:]))
        if not pairs:
            break
        top_pair = pairs.most_common(1)[0][0]
        new_token = ''.join(top_pair)
        tokens = merge_pair(tokens, top_pair, new_token)
        vocab.append(new_token)
        merges.append(top_pair)

    return vocab, merges

def encode(text, merges, vocab):
    """Encode a string to a list of token IDs.

    Works for both BPE (pass merge rules) and char-level (pass merges=[]).
    Applies merge rules in order, then maps each token to its vocab index.
    """
    tokens = list(text)
    for pair in merges:
        tokens = merge_pair(tokens, pair, ''.join(pair))
    token_to_id = {t: i for i, t in enumerate(vocab)}
    return [token_to_id[t] for t in tokens if t in token_to_id]

def decode(ids, vocab):
    """Decode a list of token IDs back to a string."""
    return ''.join(vocab[i] for i in ids)

def merge_pair(tokens, pair, new_token):
    """Replace all non-overlapping occurrences of pair in tokens with new_token."""
    merged, i = [], 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
            merged.append(new_token)
            i += 2
        else:
            merged.append(tokens[i])
            i += 1
    return merged

def load_data(train_path, val_path, n, bpe_merges=0):
    """Load pre-split data files and tokenize, returning train/val chunk tensors.

    Args:
        train_path: path to training text file
        val_path:   path to validation text file
        n:          chunk (sequence) length
        bpe_merges: number of BPE merges; 0 = char-level tokenization

    Returns:
        train_chunks, val_chunks: LongTensors of shape (n_chunks, n)
        vocab:                list of token strings (index = token ID)
        merges:               BPE merge rules (empty list for char-level)
        avg_chars_per_token:  used to convert cross-entropy loss to BPC
    """
    with open(train_path) as f: train_text = f.read()
    with open(val_path)   as f: val_text   = f.read()

    # Vocab and BPE merges are learned from training data only;
    # base vocab is always full ASCII so IDs are stable across corpora.
    if bpe_merges > 0:
        print(f"Training BPE with {bpe_merges} merges...")
        vocab, merges = train_bpe(train_text, bpe_merges)
    else:
        vocab = ASCII_VOCAB[:]
        merges = []

    print(f"Vocab size: {len(vocab)}")
    avg_chars_per_token = len(train_text) / len(encode(train_text, merges, vocab))
    print(f"Avg chars/token: {avg_chars_per_token:.3f}")

    def to_chunks(text):
        ids = encode(text, merges, vocab)
        tokens = torch.tensor(ids)
        n_chunks = len(tokens) // n
        return tokens[:n_chunks * n].view(n_chunks, n)

    return to_chunks(train_text), to_chunks(val_text), vocab, merges, avg_chars_per_token


In [10]:
import os
import math
import csv
from datetime import datetime

RUNS_FILE = "training_runs.txt"
CSV_FILE  = "training_log.csv"

CSV_HEADER = [
    "name", "timestamp", "train_loss", "full_val_loss", "tiny_val_loss", "epochs",
    "layers", "d_model", "d_ff", "H", "B", "n", "lr", "vocab_size",
    "avg_chars_per_token", "full_val_bpc", "tiny_val_bpc", "time_per_epoch_s", "total_time_s", "notes"
]

def log_results(cfg, train_loss, val_loss, tiny_val_loss=None, avg_chars_per_token=1.0,
                epoch_losses=None, time_per_epoch=0, total_time=0, note="", name="",
                runs_path=RUNS_FILE, csv_path=CSV_FILE):
    """Append a training run summary to the text log and CSV.

    BPC (bits per character) = cross-entropy loss / log(2) / avg_chars_per_token.
    val_loss / val_bpc come from full_val (per-epoch monitor).
    tiny_val_loss / tiny_val_bpc come from tiny_val (cross-dataset benchmark).
    """
    bpc_train    = train_loss / math.log(2) / avg_chars_per_token
    bpc_val      = val_loss   / math.log(2) / avg_chars_per_token
    bpc_tiny_val = tiny_val_loss / math.log(2) / avg_chars_per_token if tiny_val_loss is not None else None
    timestamp    = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    lines = []
    lines.append(f"\n{'='*60}")
    lines.append(f"Run: {timestamp}" + (f"  [{name}]" if name else ""))
    lines.append(f"{'='*60}")
    lines.append(f"layers={cfg.layers}, d_model={cfg.d_model}, d_ff={cfg.d_ff}, H={cfg.H}")
    lines.append(f"B={cfg.B}, n={cfg.n}, lr={cfg.lr}, vocab_size={cfg.vocab_size}")
    lines.append(f"avg_chars_per_token={avg_chars_per_token:.3f}")
    if note:
        lines.append(f"Note: {note}")
    lines.append("")

    if epoch_losses:
        lines.append(f"{'Epoch':>6}  {'train':>8}  {'val':>8}  {'time':>7}")
        lines.append("-" * 38)
        for ep, (tl, vl, et) in enumerate(epoch_losses):
            lines.append(f"{ep:>6}  {tl:>8.4f}  {vl:>8.4f}  {et:>6.1f}s")
        lines.append("")

    lines.append(f"Final  | train:     {train_loss:.4f}  bpc: {bpc_train:.4f}")
    lines.append(f"       | full_val:  {val_loss:.4f}  bpc: {bpc_val:.4f}")
    if tiny_val_loss is not None:
        lines.append(f"       | tiny_val:  {tiny_val_loss:.4f}  bpc: {bpc_tiny_val:.4f}")
    lines.append(f"Time   | total: {total_time:.1f}s  avg/epoch: {time_per_epoch:.1f}s")

    text = "\n".join(lines) + "\n"
    with open(runs_path, "a") as f:
        f.write(text)
    print(text)
    print(f"Logged to {runs_path}")

    write_header = not os.path.exists(csv_path)
    with open(csv_path, "a", newline="") as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(CSV_HEADER)
        tiny_val_loss_r = round(tiny_val_loss, 6) if tiny_val_loss is not None else ""
        tiny_val_bpc_r  = round(bpc_tiny_val, 3)  if bpc_tiny_val is not None else ""
        writer.writerow([
            name,
            datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
            round(train_loss, 6), round(val_loss, 6), tiny_val_loss_r, cfg.epochs,
            cfg.layers, cfg.d_model, cfg.d_ff, cfg.H, cfg.B, cfg.n,
            cfg.lr, cfg.vocab_size,
            round(avg_chars_per_token, 3), round(bpc_val, 3), tiny_val_bpc_r,
            round(time_per_epoch, 1), round(total_time, 1),
            note
        ])
    print(f"CSV row appended to {csv_path}")


In [11]:
import torch
import math


class Transformer():
    """Decoder-only Transformer implemented as a plain Python class.

    No nn.Module. Parameters are raw torch.Tensors stored as instance attributes.
    All operations (attention, layernorm, GELU, etc.) are written out explicitly.
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        d = cfg.device

        # Small init scale (0.02) keeps activations in a stable range at the start of training
        self.W_emb = (torch.randn(cfg.vocab_size, cfg.d_model, device=d) * 0.02).requires_grad_(True)

        # LayerNorm parameters: gamma (scale) initialized to 1, beta (shift) to 0
        self.gamma_out   = torch.ones(cfg.d_model, device=d).requires_grad_(True)
        self.beta_out    = torch.zeros(cfg.d_model, device=d).requires_grad_(True)
        self.gamma_attn  = [torch.ones(cfg.d_model, device=d).requires_grad_(True) for _ in range(cfg.layers)]
        self.beta_attn   = [torch.zeros(cfg.d_model, device=d).requires_grad_(True) for _ in range(cfg.layers)]
        self.gamma_ffn   = [torch.ones(cfg.d_model, device=d).requires_grad_(True) for _ in range(cfg.layers)]
        self.beta_ffn    = [torch.zeros(cfg.d_model, device=d).requires_grad_(True) for _ in range(cfg.layers)]

        self.Wq  = [(torch.randn(cfg.d_model, cfg.d_model, device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Wk  = [(torch.randn(cfg.d_model, cfg.d_model, device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Wv  = [(torch.randn(cfg.d_model, cfg.d_model, device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Wo  = [(torch.randn(cfg.d_model, cfg.d_model, device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Wff1 = [(torch.randn(cfg.d_model, cfg.d_ff,  device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Wff2 = [(torch.randn(cfg.d_ff, cfg.d_model,  device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.training = True

        # Precompute RoPE rotation angles for all positions and dimension pairs.
        # thetas[j] = 1 / 10000^(2j/d_k) — lower frequency for higher dimensions.
        # angles[m, j] = m * thetas[j] — position m, dimension pair j.
        thetas = 1 / 10000 ** (torch.arange(0, cfg.d_k, 2) / cfg.d_k)
        M = torch.arange(cfg.n)
        angles = torch.outer(M, thetas)  # (n, d_k/2)
        self.sin = angles.sin().to(d)
        self.cos = angles.cos().to(d)

    def params(self):
        """Return all learnable parameters as a flat list (used by the optimizer and save/load)."""
        return [
            self.W_emb,
            *self.Wq, *self.Wk, *self.Wv, *self.Wo,
            *self.Wff1, *self.Wff2,
            *self.gamma_attn, *self.beta_attn,
            *self.gamma_ffn, *self.beta_ffn,
            self.gamma_out, self.beta_out
        ]

    def layerNorm(self, X, gamma, beta):
        """Pre-norm LayerNorm: normalize over the last dimension, then scale and shift."""
        X_means = X.mean(dim=-1, keepdim=True)
        X_var = X.var(dim=-1, keepdim=True, unbiased=False)
        X_norm = (X - X_means) / torch.sqrt(X_var + 1e-5)  # 1e-5 for numerical stability
        return gamma * X_norm + beta

    def RoPE(self, X):
        """Apply Rotary Position Embedding to X ~ (B, H, n, d_k).

        Splits d_k into even/odd index pairs and rotates each pair by a
        position-dependent angle: [x1, x2] → [x1·cos - x2·sin, x1·sin + x2·cos].
        This encodes relative position into the dot-product attention score.
        """
        x1 = X[..., 0::2]  # even indices
        x2 = X[..., 1::2]  # odd indices
        x1r = x1 * self.cos - x2 * self.sin
        x2r = x1 * self.sin + x2 * self.cos
        # Interleave rotated pairs back into the original dimension order
        Xr = torch.stack([x1r, x2r], dim=-1)
        Xr = Xr.flatten(-2)
        return Xr

    def dropout(self, X):
        """Inverted dropout: zero out drop_prob fraction of units and scale up survivors.

        Scaling by 1/(1-p) at train time keeps the expected magnitude unchanged,
        so inference can run with no adjustment (just return X when not training).
        """
        if not self.training:
            return X
        prob = self.cfg.drop_prob
        mask = (torch.rand(X.shape, device=X.device) > prob).float()
        mask = mask / (1 - prob)
        return X * mask

    def attention(self, X, i):
        """Multi-head causal self-attention for layer i."""
        cfg = self.cfg

        Q = X @ self.Wq[i]
        K = X @ self.Wk[i]
        V = X @ self.Wv[i]

        # Reshape from (B, n, d_model) → (B, H, n, d_k) so each head operates independently
        Q = Q.view(cfg.B, cfg.n, cfg.H, cfg.d_k).transpose(-3, -2)
        K = K.view(cfg.B, cfg.n, cfg.H, cfg.d_k).transpose(-3, -2)
        V = V.view(cfg.B, cfg.n, cfg.H, cfg.d_v).transpose(-3, -2)

        Q = self.RoPE(Q)
        K = self.RoPE(K)

        S = Q @ K.transpose(-2, -1)
        S_scaled = S / math.sqrt(cfg.d_k)

        # Causal mask: diagonal=1 leaves the main diagonal (self-attention) unmasked
        mask = torch.triu(torch.full((cfg.n, cfg.n), float('-inf'), device=cfg.device), diagonal=1)
        S_masked = S_scaled + mask

        A = self.dropout(torch.softmax(S_masked, dim=-1))
        O = A @ V

        # Recombine heads: (B, H, n, d_v) → (B, n, d_model)
        O = O.transpose(-3, -2).reshape(cfg.B, cfg.n, cfg.H * cfg.d_v)
        X_out = O @ self.Wo[i]
        return X_out

    def FeedForwardNetwork(self, X, i):
        """Two-layer FFN with GELU activation for layer i."""
        Z1 = X @ self.Wff1[i]
        # GELU (tanh approximation) — smoother than ReLU, standard in GPT-family models
        Z1_GELU = 0.5 * Z1 * (1 + torch.tanh(math.sqrt(2 / math.pi) * (Z1 + 0.044715 * Z1 ** 3)))
        X_out = Z1_GELU @ self.Wff2[i]
        return X_out

    def CrossEntropyLoss(self, logits, targets):
        """Numerically stable cross-entropy loss.

        Subtracting the row-wise max before exp prevents overflow without
        changing the softmax output (the constant cancels in numerator/denominator).
        """
        logits = logits.reshape(-1, self.cfg.vocab_size)
        targets = targets.reshape(-1)

        l_stable = logits - logits.max(dim=1, keepdim=True).values
        exp = torch.exp(l_stable)
        l_softmax = exp / torch.sum(exp, dim=1, keepdim=True)
        # Select the softmax probability of the correct class for each token
        p_c = l_softmax[torch.arange(len(targets), device=self.cfg.device), targets]
        loss = -torch.sum(torch.log(p_c)) / len(logits)
        return loss

    def forward_pass(self, tokens):
        """Full forward pass: embedding → transformer blocks → output logits + targets.

        The output projection reuses W_emb transposed (weight tying), which reduces
        parameters and ties the input and output token representations together.
        Logits and targets are shifted by one position: token t predicts token t+1.
        """
        X = self.W_emb[tokens]  # (B, n) → (B, n, d_model)
        for i in range(self.cfg.layers):
            X = X + self.dropout(self.attention(self.layerNorm(X, self.gamma_attn[i], self.beta_attn[i]), i))
            X = X + self.dropout(self.FeedForwardNetwork(self.layerNorm(X, self.gamma_ffn[i], self.beta_ffn[i]), i))
        X = self.layerNorm(X, self.gamma_out, self.beta_out)

        logits = X @ self.W_emb.T          # weight tying: reuse embedding matrix as output projection
        logits = logits[:, :-1, :]         # drop prediction at last position (no target for it)
        targets = tokens[:, 1:]            # shift targets left by 1: position t predicts token t+1
        return logits, targets

In [ ]:
import torch
import math


class LoRATransformer():
    """Decoder-only Transformer implemented as a plain Python class.

    No nn.Module. Parameters are raw torch.Tensors stored as instance attributes.
    All operations (attention, layernorm, GELU, etc.) are written out explicitly.
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        d = cfg.device

        # Small init scale (0.02) keeps activations in a stable range at the start of training
        self.W_emb = (torch.randn(cfg.vocab_size, cfg.d_model, device=d) * 0.02).requires_grad_(False)

        # LayerNorm parameters: gamma (scale) initialized to 1, beta (shift) to 0
        self.gamma_out   = torch.ones(cfg.d_model, device=d).requires_grad_(False)
        self.beta_out    = torch.zeros(cfg.d_model, device=d).requires_grad_(False)
        self.gamma_attn  = [torch.ones(cfg.d_model, device=d).requires_grad_(False) for _ in range(cfg.layers)]
        self.beta_attn   = [torch.zeros(cfg.d_model, device=d).requires_grad_(False) for _ in range(cfg.layers)]
        self.gamma_ffn   = [torch.ones(cfg.d_model, device=d).requires_grad_(False) for _ in range(cfg.layers)]
        self.beta_ffn    = [torch.zeros(cfg.d_model, device=d).requires_grad_(False) for _ in range(cfg.layers)]

        self.Wq  = [(torch.randn(cfg.d_model, cfg.d_model, device=d) * 0.02).requires_grad_(False) for _ in range(cfg.layers)]
        self.Wk  = [(torch.randn(cfg.d_model, cfg.d_model, device=d) * 0.02).requires_grad_(False) for _ in range(cfg.layers)]
        self.Wv  = [(torch.randn(cfg.d_model, cfg.d_model, device=d) * 0.02).requires_grad_(False) for _ in range(cfg.layers)]
        self.Wo  = [(torch.randn(cfg.d_model, cfg.d_model, device=d) * 0.02).requires_grad_(False) for _ in range(cfg.layers)]
        self.Wff1 = [(torch.randn(cfg.d_model, cfg.d_ff,  device=d) * 0.02).requires_grad_(False) for _ in range(cfg.layers)]
        self.Wff2 = [(torch.randn(cfg.d_ff, cfg.d_model,  device=d) * 0.02).requires_grad_(False) for _ in range(cfg.layers)]
        
        self.Aq  = [(torch.randn(cfg.d_model, cfg.lora_rank, device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Ak  = [(torch.randn(cfg.d_model, cfg.lora_rank, device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Av  = [(torch.randn(cfg.d_model, cfg.lora_rank, device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Ao  = [(torch.randn(cfg.d_model, cfg.lora_rank, device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Aff1 = [(torch.randn(cfg.d_model, cfg.lora_rank,  device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Aff2 = [(torch.randn(cfg.d_ff, cfg.lora_rank,  device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        
        self.Bq  = [(torch.zeros(cfg.lora_rank, cfg.d_model, device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Bk  = [(torch.zeros(cfg.lora_rank, cfg.d_model, device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Bv  = [(torch.zeros(cfg.lora_rank, cfg.d_model, device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Bo  = [(torch.zeros(cfg.lora_rank, cfg.d_model, device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Bff1 = [(torch.zeros(cfg.lora_rank, cfg.d_ff,  device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        self.Bff2 = [(torch.zeros(cfg.lora_rank, cfg.d_model,  device=d) * 0.02).requires_grad_(True) for _ in range(cfg.layers)]
        
        self.training = True

        # Precompute RoPE rotation angles for all positions and dimension pairs.
        # thetas[j] = 1 / 10000^(2j/d_k) — lower frequency for higher dimensions.
        # angles[m, j] = m * thetas[j] — position m, dimension pair j.
        thetas = 1 / 10000 ** (torch.arange(0, cfg.d_k, 2) / cfg.d_k)
        M = torch.arange(cfg.n)
        angles = torch.outer(M, thetas)  # (n, d_k/2)
        self.sin = angles.sin().to(d)
        self.cos = angles.cos().to(d)

    def params(self):
        """Return all learnable parameters as a flat list (used by the optimizer and save/load)."""
        return [
            *self.Aq, *self.Ak, *self.Av, *self.Ao,
            *self.Aff1, *self.Aff2,
            *self.Bq, *self.Bk, *self.Bv, *self.Bo,
            *self.Bff1, *self.Bff2,
        ]

    def layerNorm(self, X, gamma, beta):
        """Pre-norm LayerNorm: normalize over the last dimension, then scale and shift."""
        X_means = X.mean(dim=-1, keepdim=True)
        X_var = X.var(dim=-1, keepdim=True, unbiased=False)
        X_norm = (X - X_means) / torch.sqrt(X_var + 1e-5)  # 1e-5 for numerical stability
        return gamma * X_norm + beta

    def RoPE(self, X):
        """Apply Rotary Position Embedding to X ~ (B, H, n, d_k).

        Splits d_k into even/odd index pairs and rotates each pair by a
        position-dependent angle: [x1, x2] → [x1·cos - x2·sin, x1·sin + x2·cos].
        This encodes relative position into the dot-product attention score.
        """
        x1 = X[..., 0::2]  # even indices
        x2 = X[..., 1::2]  # odd indices
        x1r = x1 * self.cos - x2 * self.sin
        x2r = x1 * self.sin + x2 * self.cos
        # Interleave rotated pairs back into the original dimension order
        Xr = torch.stack([x1r, x2r], dim=-1)
        Xr = Xr.flatten(-2)
        return Xr

    def dropout(self, X):
        """Inverted dropout: zero out drop_prob fraction of units and scale up survivors.

        Scaling by 1/(1-p) at train time keeps the expected magnitude unchanged,
        so inference can run with no adjustment (just return X when not training).
        """
        if not self.training:
            return X
        prob = self.cfg.drop_prob
        mask = (torch.rand(X.shape, device=X.device) > prob).float()
        mask = mask / (1 - prob)
        return X * mask

    def attention(self, X, i):
        """Multi-head causal self-attention for layer i."""
        cfg = self.cfg

        Q = X @ self.Wq[i] + (cfg.lora_alpha / cfg.lora_rank) * (X @ self.Aq[i]) @ self.Bq[i]
        K = X @ self.Wk[i] + (cfg.lora_alpha / cfg.lora_rank) * (X @ self.Ak[i]) @ self.Bk[i]
        V = X @ self.Wv[i] + (cfg.lora_alpha / cfg.lora_rank) * (X @ self.Av[i]) @ self.Bv[i]


        # Reshape from (B, n, d_model) → (B, H, n, d_k) so each head operates independently
        Q = Q.view(cfg.B, cfg.n, cfg.H, cfg.d_k).transpose(-3, -2)
        K = K.view(cfg.B, cfg.n, cfg.H, cfg.d_k).transpose(-3, -2)
        V = V.view(cfg.B, cfg.n, cfg.H, cfg.d_v).transpose(-3, -2)

        Q = self.RoPE(Q)
        K = self.RoPE(K)

        S = Q @ K.transpose(-2, -1)
        S_scaled = S / math.sqrt(cfg.d_k)

        # Causal mask: diagonal=1 leaves the main diagonal (self-attention) unmasked
        mask = torch.triu(torch.full((cfg.n, cfg.n), float('-inf'), device=cfg.device), diagonal=1)
        S_masked = S_scaled + mask

        A = self.dropout(torch.softmax(S_masked, dim=-1))
        O = A @ V

        # Recombine heads: (B, H, n, d_v) → (B, n, d_model)
        O = O.transpose(-3, -2).reshape(cfg.B, cfg.n, cfg.H * cfg.d_v)
        X_out = O @ self.Wo[i] + (cfg.lora_alpha / cfg.lora_rank) * (O @ self.Ao[i]) @ self.Bo[i]
        return X_out

    def FeedForwardNetwork(self, X, i):
        cfg = self.cfg
        """Two-layer FFN with GELU activation for layer i."""
        Z1 = X @ self.Wff1[i] + (cfg.lora_alpha / cfg.lora_rank) * (X @ self.Aff1[i]) @ self.Bff1[i]
        # GELU (tanh approximation) — smoother than ReLU, standard in GPT-family models
        Z1_GELU = 0.5 * Z1 * (1 + torch.tanh(math.sqrt(2 / math.pi) * (Z1 + 0.044715 * Z1 ** 3)))
        X_out = Z1_GELU @ self.Wff2[i] + (cfg.lora_alpha / cfg.lora_rank) * (Z1_GELU @ self.Aff2[i]) @ self.Bff2[i]
        return X_out

    def CrossEntropyLoss(self, logits, targets):
        """Numerically stable cross-entropy loss.

        Subtracting the row-wise max before exp prevents overflow without
        changing the softmax output (the constant cancels in numerator/denominator).
        """
        logits = logits.reshape(-1, self.cfg.vocab_size)
        targets = targets.reshape(-1)

        l_stable = logits - logits.max(dim=1, keepdim=True).values
        exp = torch.exp(l_stable)
        l_softmax = exp / torch.sum(exp, dim=1, keepdim=True)
        # Select the softmax probability of the correct class for each token
        p_c = l_softmax[torch.arange(len(targets), device=self.cfg.device), targets]
        loss = -torch.sum(torch.log(p_c)) / len(logits)
        return loss

    def forward_pass(self, tokens):
        """Full forward pass: embedding → transformer blocks → output logits + targets.

        The output projection reuses W_emb transposed (weight tying), which reduces
        parameters and ties the input and output token representations together.
        Logits and targets are shifted by one position: token t predicts token t+1.
        """
        X = self.W_emb[tokens]  # (B, n) → (B, n, d_model)
        for i in range(self.cfg.layers):
            X = X + self.dropout(self.attention(self.layerNorm(X, self.gamma_attn[i], self.beta_attn[i]), i))
            X = X + self.dropout(self.FeedForwardNetwork(self.layerNorm(X, self.gamma_ffn[i], self.beta_ffn[i]), i))
        X = self.layerNorm(X, self.gamma_out, self.beta_out)

        logits = X @ self.W_emb.T          # weight tying: reuse embedding matrix as output projection
        logits = logits[:, :-1, :]         # drop prediction at last position (no target for it)
        targets = tokens[:, 1:]            # shift targets left by 1: position t predicts token t+1
        return logits, targets

In [ ]:
# ── Load pretrained weights into LoRATransformer ─────────────────────────────
# Frozen params are copied from a base Transformer checkpoint.
# B matrices are already zeroed at init, so the LoRA correction (α/r · A@B)
# starts at 0 — model outputs are identical to the pretrained base at step 0.

cfg = Config()
cfg.vocab_size = len(vocab)  # assumes vocab already loaded in session

lora_model = LoRATransformer(cfg)

# Load pretrained weights into a temporary base model, then copy across
base_model = Transformer(cfg)
load_model(base_model, path="checkpoints/v1-9_brash-tarn.pt")

lora_model.W_emb.data.copy_(base_model.W_emb.data)
lora_model.gamma_out.data.copy_(base_model.gamma_out.data)
lora_model.beta_out.data.copy_(base_model.beta_out.data)

for i in range(cfg.layers):
    lora_model.Wq[i].data.copy_(base_model.Wq[i].data)
    lora_model.Wk[i].data.copy_(base_model.Wk[i].data)
    lora_model.Wv[i].data.copy_(base_model.Wv[i].data)
    lora_model.Wo[i].data.copy_(base_model.Wo[i].data)
    lora_model.Wff1[i].data.copy_(base_model.Wff1[i].data)
    lora_model.Wff2[i].data.copy_(base_model.Wff2[i].data)
    lora_model.gamma_attn[i].data.copy_(base_model.gamma_attn[i].data)
    lora_model.beta_attn[i].data.copy_(base_model.beta_attn[i].data)
    lora_model.gamma_ffn[i].data.copy_(base_model.gamma_ffn[i].data)
    lora_model.beta_ffn[i].data.copy_(base_model.beta_ffn[i].data)

del base_model  # free memory

lora_opt = torch.optim.AdamW(lora_model.params(), lr=cfg.lr, weight_decay=0.01)
print(f"LoRA trainable params: {sum(p.numel() for p in lora_model.params()):,}")
print("Ready to fine-tune.")

In [12]:
import time
import os
import glob
import json
import random

_ADJECTIVES = [
    "amber","azure","bold","brash","bright","brine","bronze","calm","crisp",
    "dapper","dark","dawn","deft","dense","dusk","feral","fierce","frosty",
    "gilded","grim","hollow","jade","lithe","lunar","mossy","murky","noble",
    "obsidian","pale","prim","quaint","quiet","rapid","raven","rugged","russet",
    "sage","scarlet","serene","sharp","silver","sleek","slim","stark","stern",
    "stout","sunlit","swift","tawny","terse","tidal","torrid","trite","twilight",
    "velvet","vivid","wan","wild","wiry","worn","zeal","zenith","zephyr",
    "quixotic","languid","stoic","furtive","lucid","brazen","placid","sullen",
]
_NOUNS = [
    "blush","brook","brow","cairn","cask","chalk","cinder","cliff","crest",
    "dell","drift","dusk","ember","fern","fjord","flint","foam","gale","glade",
    "glen","gust","haven","heath","helm","hill","hollow","kelp","knoll","lagoon",
    "lark","ledge","loch","mast","mead","mire","moor","moss","peak","peat",
    "pine","pond","pool","reed","ridge","rill","rind","roan","rock","rune",
    "rush","salt","scree","sedge","shoal","shrub","silt","slate","sleet","slope",
    "snow","spar","spire","spray","sprig","spur","stone","storm","strath","stream",
    "surge","swale","tarn","thatch","tide","tor","vale","vault","wake","weld","wren",
]

def _next_version():
    import re
    existing = glob.glob("checkpoints/v*_*.pt")
    nums = [int(m.group(1)) for f in existing
            for m in [re.search(r'v1-(\d+)_', f)] if m]
    return f"1-{max(nums) + 1 if nums else 0}"

def _make_run_name():
    return f"v{_next_version()}_{random.choice(_ADJECTIVES)}-{random.choice(_NOUNS)}"

def save_model(model, cfg, vocab, merges=None, directory="checkpoints", name=None):
    """Save model weights and artifacts. Returns (path, name)."""
    os.makedirs(directory, exist_ok=True)
    if name is None:
        name = _make_run_name()
    base = os.path.join(directory, name)
    torch.save([p.detach().cpu() for p in model.params()], base + ".pt")
    cfg_dict = {k: v for k, v in cfg.__dict__.items() if k != "device"}
    with open(base + "_cfg.json", "w") as f:
        json.dump(cfg_dict, f)
    with open(base + "_vocab.json", "w") as f:
        json.dump(vocab, f)
    if merges:
        with open(base + "_merges.json", "w") as f:
            json.dump([list(pair) for pair in merges], f)
    print(f"Saved model to {base}.pt")
    return base + ".pt", name

def load_model(model, path=None, directory="checkpoints"):
    """Load weights from a checkpoint. If path is None, loads the latest."""
    if path is None:
        files = sorted(glob.glob(os.path.join(directory, "v*.pt")))
        if not files:
            raise FileNotFoundError(f"No checkpoints found in '{directory}/'")
        path = files[-1]
    saved_params = torch.load(path, map_location=model.cfg.device, weights_only=True)
    for p, saved in zip(model.params(), saved_params):
        p.data.copy_(saved.to(model.cfg.device))
    print(f"Loaded model from {path}")
    return path

def load_for_inference(weights=None, directory="checkpoints"):
    """Reconstruct model + tokenizer from a checkpoint, with no prior setup needed."""
    if weights is None:
        files = sorted(glob.glob(os.path.join(directory, "v*.pt")))
        if not files:
            raise FileNotFoundError(f"No checkpoints found in '{directory}/'")
        weights = files[-1]
    base = weights[:-3]
    with open(base + "_cfg.json") as f:
        cfg_dict = json.load(f)
    with open(base + "_vocab.json") as f:
        vocab = json.load(f)
    merges_path = base + "_merges.json"
    merges = []
    if os.path.exists(merges_path):
        with open(merges_path) as f:
            merges = [tuple(pair) for pair in json.load(f)]
    cfg = Config(**{k: v for k, v in cfg_dict.items() if k in Config.__dataclass_fields__})
    model = Transformer(cfg)
    load_model(model, path=weights)
    return model, vocab, merges

def eval_loss(model, chunks, cfg):
    """Evaluate average cross-entropy loss over all complete batches in chunks."""
    model.training = False
    n_batches = chunks.shape[0] // cfg.B
    batches = chunks[:n_batches * cfg.B].view(n_batches, cfg.B, cfg.n)
    total_loss = 0
    with torch.no_grad():
        for batch in batches:
            logits, targets = model.forward_pass(batch.to(cfg.device))
            total_loss += model.CrossEntropyLoss(logits, targets).item()
    model.training = True
    return total_loss / n_batches

def save_params(model):
    """Snapshot all model parameters to CPU tensors."""
    return [p.detach().clone() for p in model.params()]

def load_params(model, snapshot):
    """Restore model parameters from a snapshot."""
    for p, saved in zip(model.params(), snapshot):
        p.data.copy_(saved)

def train(model, optimizer, train_chunks, val_chunks, cfg, vocab, merges=None,
          avg_chars_per_token=1.0, patience=5, note="", scheduler=None,
          tiny_val_chunks=None):
    """Train with early stopping and best-checkpoint restoration.

    Shuffles training chunks each epoch, evaluates train/val loss after each epoch,
    snapshots best-val weights in memory, and restores them at the end.
    Evaluates tiny_val (if provided) after restoring the best checkpoint.
    Logs results and saves a named checkpoint file.
    """
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    train_loss = val_loss = None
    completed_epochs = 0
    epoch_losses = []
    total_start = time.time()
    best_snapshot = save_params(model)
    run_name = _make_run_name()

    try:
        for epoch in range(cfg.epochs):
            epoch_start = time.time()
            chunks = train_chunks[torch.randperm(train_chunks.shape[0])]
            n_batches = chunks.shape[0] // cfg.B
            batches = chunks[:n_batches * cfg.B].view(n_batches, cfg.B, cfg.n)
            for batch_idx, batch in enumerate(batches):
                logits, targets = model.forward_pass(batch.to(cfg.device))
                loss = model.CrossEntropyLoss(logits, targets)
                loss.backward()
                print(f"Epoch {epoch} | Loss {loss.item():.4f} | Batch {batch_idx + 1} / {n_batches}", end='\r')
                torch.nn.utils.clip_grad_norm_(model.params(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

            train_loss = eval_loss(model, train_chunks, cfg)
            val_loss   = eval_loss(model, val_chunks,   cfg)
            epoch_time = time.time() - epoch_start
            completed_epochs += 1
            epoch_losses.append((train_loss, val_loss, epoch_time))
            print(f"Epoch {epoch} | train: {train_loss:.4f} | val: {val_loss:.4f} | {epoch_time:.1f}s")

            if scheduler:
                scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_without_improvement = 0
                best_snapshot = save_params(model)
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break

    except KeyboardInterrupt:
        print(f"\nStopped at epoch {epoch} (completed {completed_epochs} full epochs)")
        if completed_epochs == 0:
            print("No complete epochs — nothing to log.")
            return None, None, None

    # Restore best-val checkpoint
    load_params(model, best_snapshot)
    train_loss = eval_loss(model, train_chunks, cfg)
    val_loss   = eval_loss(model, val_chunks,   cfg)
    tiny_val_loss = eval_loss(model, tiny_val_chunks, cfg) if tiny_val_chunks is not None else None

    total_time     = time.time() - total_start
    avg_epoch_time = sum(et for _, _, et in epoch_losses) / len(epoch_losses) if epoch_losses else 0
    cfg.epochs     = completed_epochs

    _, run_name = save_model(model, cfg, vocab, merges, name=run_name)
    log_results(cfg, train_loss, val_loss, tiny_val_loss=tiny_val_loss,
                avg_chars_per_token=avg_chars_per_token, epoch_losses=epoch_losses,
                time_per_epoch=avg_epoch_time, total_time=total_time, note=note, name=run_name)
    return train_loss, val_loss, tiny_val_loss


In [13]:
# ── Run experiments sequentially ─────────────────────────────────────────────
# All three run in one cell so you can close the browser after starting.
# Vocab/merges come from full_train (largest vocabulary) so tokenization is
# identical across all runs. avg_chars_per_token is per-corpus for accurate BPC.

cfg = Config()

experiments = [
 #   ("data/full_train_minus_tiny_val.txt",
 #    "corpus=full_train_minus_tiny_val (3.5M chars, t8, tiny_val overlap removed)"),
    ("data/full_train_862k.txt",
     "corpus=full_train_862k (862k chars, t8 edition, different scenes from tiny_train)"),
    ("data/tiny_train_t8.txt",
     "corpus=tiny_train_t8 (862K chars, t8 edition, same scenes as tiny_train)"),
]

def _encode_file(path, merges, vocab, n):
    """Encode a text file with existing vocab/merges, return (chunks, avg_chars_per_token)."""
    with open(path) as f: text = f.read()
    ids = encode(text, merges, vocab)
    tokens = torch.tensor(ids)
    n_chunks = len(tokens) // n
    avg_cpt = len(text) / max(len(ids), 1)
    return tokens[:n_chunks * n].view(n_chunks, n), avg_cpt

# Build vocab/merges once from full_train (broadest vocabulary)
_, _, vocab_exp, merges_exp, _ = load_data(
    "data/full_train.txt", "data/full_val.txt", n=cfg.n, bpe_merges=cfg.bpe_merges)

# Encode val sets once with vocab_exp
val_chunks_exp, _ = _encode_file("data/full_val.txt", merges_exp, vocab_exp, cfg.n)
tiny_val_exp, _   = _encode_file("data/tiny_val.txt", merges_exp, vocab_exp, cfg.n)

for train_path, note in experiments:
    print(f"\n{'='*60}\nStarting: {note}\n{'='*60}")
    train_chunks_exp, avg_cpt = _encode_file(train_path, merges_exp, vocab_exp, cfg.n)
    cfg_exp = Config()
    cfg_exp.vocab_size = len(vocab_exp)
    model_exp = Transformer(cfg_exp)
    opt_exp = torch.optim.AdamW(model_exp.params(), lr=cfg_exp.lr, weight_decay=0.01)
    sched_exp = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_exp, factor=0.5, patience=2)
    train(model_exp, opt_exp, train_chunks_exp, val_chunks_exp, cfg_exp,
          vocab=vocab_exp, merges=merges_exp, avg_chars_per_token=avg_cpt,
          scheduler=sched_exp, note=note, tiny_val_chunks=tiny_val_exp)


Using device: cuda
Vocab size: 128
Avg chars/token: 1.000

Starting: corpus=full_train_862k (862k chars, t8 edition, different scenes from tiny_train)
Using device: cuda
Epoch 0 | Loss 3.2069 | Batch 13 / 105
Stopped at epoch 0 (completed 0 full epochs)
No complete epochs — nothing to log.

Starting: corpus=tiny_train_t8 (862K chars, t8 edition, same scenes as tiny_train)
Using device: cuda
Epoch 0 | Loss 2.5328 | Batch 48 / 105
Stopped at epoch 0 (completed 0 full epochs)
No complete epochs — nothing to log.


In [18]:
def generate(model=None, vocab=None, merges=None, prompt="\n", max_new_tokens=500,
             temperature=1.0, top_k=None, weights=None):
    """
    Autoregressively sample from the model.

    Can be called with a model already in memory, or standalone from just a checkpoint:
        generate(model, vocab, merges, prompt="...")          # model in memory
        generate(prompt="...", weights="checkpoints/run_X.pt")  # no model needed

    Args:
        model:          Transformer instance (optional — reconstructed from checkpoint if None)
        vocab:          list of tokens, index = token ID (optional — loaded from checkpoint if None)
        merges:         BPE merge rules (optional — [] for char-level, loaded from checkpoint if None)
        prompt:         starting string
        max_new_tokens: number of tokens to generate
        temperature:    >1 = more random, <1 = more greedy
        top_k:          if set, only sample from the top-k most likely tokens
        weights:        path to checkpoint; if None, uses the latest in checkpoints/
    """
    if model is None:
        model, vocab, merges = load_for_inference(weights=weights)
    else:
        if weights is not None:
            load_model(model, path=weights)

    if merges is None:
        merges = []

    model.training = False
    cfg = model.cfg
    orig_B = cfg.B
    cfg.B = 1  # attention uses cfg.B for reshaping; set to 1 for single-sequence inference

    tokens = encode(prompt, merges, vocab) or [0]

    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = tokens[-cfg.n:]
            # Left-pad with token 0 if the context is shorter than the full window.
            # The model was trained on dense n-length sequences, so padding warm-up
            # degrades output quality until real tokens fill the context.
            if len(context) < cfg.n:
                context = [0] * (cfg.n - len(context)) + context

            x = torch.tensor([context], device=cfg.device)
            X = model.W_emb[x]
            for i in range(cfg.layers):
                X = X + model.attention(model.layerNorm(X, model.gamma_attn[i], model.beta_attn[i]), i)
                X = X + model.FeedForwardNetwork(model.layerNorm(X, model.gamma_ffn[i], model.beta_ffn[i]), i)
            X = model.layerNorm(X, model.gamma_out, model.beta_out)
            logit = (X @ model.W_emb.T)[0, -1, :]  # take only the last-position logit

            logit = logit / temperature
            if top_k is not None:
                # Mask all tokens below the k-th highest logit before softmax
                threshold = torch.topk(logit, top_k).values[-1]
                logit[logit < threshold] = float('-inf')

            probs = torch.softmax(logit, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            tokens.append(next_token)

    cfg.B = orig_B
    model.training = True
    return decode(tokens, vocab)


prompt = """SEBASTIAN:
You were kneel'd to and importuned otherwise
By all of us, and the fair soul herself
Weigh'd between loathness and obedience, at
Which end o' the beam should bow. We have lost your
son,
I fear, for ever: Milan and Naples have
More widows in them of this business' making
Than we bring men to comfort them:
The fault's your own.

ALONSO:
So is the dear'st o' the loss.

GONZALO:
My lord Sebastian,
The truth you speak doth lack some gentleness
And time to speak it in: you rub the sore,
When you should bring the plaster.

SEBASTIAN:
Very well."""

print(generate(prompt=prompt, max_new_tokens=800, temperature=0.9, top_k=50, weights="checkpoints/v1-9_brash-tarn.pt"))
#print(generate(prompt="\n", max_new_tokens=500, temperature=0.8, top_k=50))
# --- standalone from checkpoint ---
# print(generate(prompt=prompt, max_new_tokens=500, temperature=0.8, top_k=30))

Using device: cuda
Loaded model from checkpoints/v1-9_brash-tarn.pt
SEBASTIAN:
You were kneel'd to and importuned otherwise
By all of us, and the fair soul herself
Weigh'd between loathness and obedience, at
Which end o' the beam should bow. We have lost your
son,
I fear, for ever: Milan and Naples have
More widows in them of this business' making
Than we bring men to comfort them:
The fault's your own.

ALONSO:
So is the dear'st o' the loss.

GONZALO:
My lord Sebastian,
The truth you speak doth lack some gentleness
And time to speak it in: you rub the sore,
When you should bring the plaster.

SEBASTIAN:
Very well.

PLANTAGENET:
Go to the way away I would have made
A man who is given learned and beseech.

PUCK:
What though good Polydore?

AUXF:
Thou art, sir, no, not for the fierce word.

AJAX:
Thou art but nettle, sir. He'll release her with the
more painted are to found- fellow. A plague of note; the
bold will prove it to my looks; of good lady, 'tis to make him
down me to th' pardon